In [1]:
import pandas as pd
import numpy as np
import glob
import os
import seaborn as sns
import matplotlib.pyplot as plt
import streamlit as st

st.set_page_config(
    page_title="Sleep & Diabetes Dashboard",
    layout="wide"
)

In [2]:
csv_folder = r'C:\Users\ruchi\OneDrive\Documents\Python Hackathon May 2026\HUPA-UC Diabetes Dataset-20250820T010637Z-1-001\HUPA-UC Diabetes Dataset'
output_file = r"C:\Temporary\combined_df_data.csv"
output_file2 = r"C:\Temporary\dashboard_data.csv"

In [3]:
files = os.path.join(csv_folder, "T1DM_patient_sleep_demographics_with_race.csv")


df = pd.read_csv(files)
output_df = pd.read_csv(output_file)


In [4]:
combined_df = pd.merge(
    df,
    output_df,
    on='Patient_ID',
    how='left'
)

## Dashboard Overview
Use the Streamlit layout below to display KPIs and charts in a responsive dashboard format.

<!-- Old KPI description cell removed: dashboard overview and metrics are handled in the Streamlit dashboard cell below. -->

In [19]:
st.title("Sleep & Diabetes Dashboard - Comprehensive Analysis")
st.markdown(
    "A comprehensive dashboard view of blood glucose control, sleep quality, sleep disturbance "
    "patterns, and insulin management across demographic groups with complete biomarker analysis."
)

# KPI Section
st.markdown("## Key Performance Indicators")
avg_glucose = combined_df['Blood_Glucose_mg_dl'].mean()
max_glucose = combined_df['Blood_Glucose_mg_dl'].max()
hyper = (combined_df['Blood_Glucose_mg_dl'] > 300).mean() * 100
avg_heart_rate = combined_df['Heart_Rate_bpm'].mean()
avg_insulin_basal = combined_df['Basal_Insulin_Rate_Unit_hr'].mean()

kpi_col1, kpi_col2, kpi_col3, kpi_col4, kpi_col5 = st.columns(5)
kpi_col1.metric("Avg Blood Glucose", f"{avg_glucose:.1f} mg/dL")
kpi_col2.metric("Max Glucose", f"{max_glucose:.0f} mg/dL")
kpi_col3.metric("Hyperglycemia %", f"{hyper:.1f}%")
kpi_col4.metric("Avg Heart Rate", f"{avg_heart_rate:.1f} bpm")
kpi_col5.metric("Avg Basal Insulin", f"{avg_insulin_basal:.2f} U/hr")

st.markdown("---")
st.markdown("## Visual Analytics")
chart_col1, chart_col2, chart_col3 = st.columns(3)

fig1, ax1 = plt.subplots(figsize=(5, 4))
sns.regplot(
    x='Average Sleep Duration (hrs)',
    y='Heart_Rate_bpm',
    data=combined_df,
    ax=ax1
)
ax1.set_title('Sleep Duration vs Heart Rate')
chart_col1.pyplot(fig1)

fig2, ax2 = plt.subplots(figsize=(5, 4))
hist_plot = sns.histplot(
    combined_df['Sleep Quality (1-10)'],
    kde=True,
    bins=5,
    ax=ax2
)
ax2.set_title('Distribution of Sleep Quality')
for bar in ax2.patches:
    height = bar.get_height()
    if height > 0:
        ax2.annotate(
            f'{int(height)}',
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 5),
            textcoords='offset points',
            ha='center',
            va='bottom'
        )
chart_col2.pyplot(fig2)

fig3, ax3 = plt.subplots(figsize=(5, 4))
bar_plot = sns.barplot(
    x='Race',
    y='% with Sleep Disturbances',
    data=combined_df,
    palette='Set2',
    ax=ax3
)
ax3.set_title('Sleep Disturbances by Race')
for bar in ax3.patches:
    height = bar.get_height()
    ax3.annotate(
        f'{height:.1f}',
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 5),
        textcoords='offset points',
        ha='center',
        va='bottom'
    )
ax3.tick_params(axis='x', rotation=30)
chart_col3.pyplot(fig3)

st.markdown("---")
st.markdown("## Comprehensive Correlation Heatmap - All Metrics")
fig4, ax4 = plt.subplots(figsize=(14, 10))
all_numeric_cols = [
    'Age',
    'Average Sleep Duration (hrs)',
    'Sleep Quality (1-10)',
    '% with Sleep Disturbances',
    'Blood_Glucose_mg_dl',
    'Basal_Insulin_Rate_Unit_hr',
    'Bolus_Insulin_Dose_Units',
    'Carbohydrate_Intake_Grams',
    'Step_count',
    'Calories_burned',
    'Heart_Rate_bpm'
]
correlation_all = combined_df[all_numeric_cols].corr()
sns.heatmap(
    correlation_all,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    ax=ax4,
    cbar_kws={'label': 'Correlation Coefficient'},
    square=True,
    linewidths=0.5
)
ax4.set_title('Complete Health Metrics Correlation Heatmap - All Biomarkers')
st.pyplot(fig4)
st.caption("Figure: Comprehensive correlation matrix of all health metrics including sleep metrics, blood glucose, insulin management, physical activity, and heart rate.")

C:\Users\ruchi\AppData\Local\Temp\ipykernel_25960\4133731625.py:58: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  bar_plot = sns.barplot(


DeltaGenerator()

In [20]:
st.markdown("---")
st.markdown("## Biomarker Correlations")

# First correlation chart - Blood Glucose and Vital Biomarkers
fig_bg, ax_bg = plt.subplots(figsize=(8, 6))
biomarker_cols_bg = [
    'Blood_Glucose_mg_dl',
    'Heart_Rate_bpm',
    'Step_count',
    'Calories_burned',
    'Age'
]
correlation_matrix_bg = combined_df[biomarker_cols_bg].corr()
sns.heatmap(
    correlation_matrix_bg,
    annot=True,
    fmt='.2f',
    cmap='RdYlBu_r',
    center=0,
    ax=ax_bg,
    cbar_kws={'label': 'Correlation Coefficient'}
)
ax_bg.set_title('Blood Glucose & Vital Biomarkers Correlation')
st.pyplot(fig_bg)
st.caption("Figure 1: Correlation matrix showing relationships between blood glucose levels and key vital biomarkers including heart rate, physical activity (steps and calories), and age.")

# Second correlation chart - Sleep and Metabolic Biomarkers
fig_sleep, ax_sleep = plt.subplots(figsize=(8, 6))
biomarker_cols_sleep = [
    'Average Sleep Duration (hrs)',
    'Sleep Quality (1-10)',
    '% with Sleep Disturbances',
    'Blood_Glucose_mg_dl',
    'Heart_Rate_bpm'
]
correlation_matrix_sleep = combined_df[biomarker_cols_sleep].corr()
sns.heatmap(
    correlation_matrix_sleep,
    annot=True,
    fmt='.2f',
    cmap='RdYlBu_r',
    center=0,
    ax=ax_sleep,
    cbar_kws={'label': 'Correlation Coefficient'}
)
ax_sleep.set_title('Sleep & Metabolic Biomarkers Correlation')
st.pyplot(fig_sleep)
st.caption("Figure 2: Correlation matrix displaying the relationships between sleep metrics (duration, quality, and disturbances) and metabolic biomarkers (blood glucose and heart rate).")

C:\Users\ruchi\AppData\Local\Temp\ipykernel_25960\4061246617.py:5: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig_bg, ax_bg = plt.subplots(figsize=(8, 6))


DeltaGenerator()

In [21]:
st.markdown("---")
st.markdown("## Additional Correlation Analysis")

# Third correlation chart - Insulin Management and Glucose Control
fig_insulin, ax_insulin = plt.subplots(figsize=(8, 6))
biomarker_cols_insulin = [
    'Blood_Glucose_mg_dl',
    'Basal_Insulin_Rate_Unit_hr',
    'Bolus_Insulin_Dose_Units',
    'Carbohydrate_Intake_Grams',
    'Age'
]
correlation_matrix_insulin = combined_df[biomarker_cols_insulin].corr()
sns.heatmap(
    correlation_matrix_insulin,
    annot=True,
    fmt='.2f',
    cmap='RdYlBu_r',
    center=0,
    ax=ax_insulin,
    cbar_kws={'label': 'Correlation Coefficient'}
)
ax_insulin.set_title('Insulin Management & Glucose Control Correlation')
st.pyplot(fig_insulin)
st.caption("Figure 3: Correlation matrix examining the relationships between insulin therapy (basal and bolus rates), blood glucose levels, carbohydrate intake, and age - critical for understanding glucose management effectiveness.")

# Fourth correlation chart - Physical Activity and Sleep Metrics
fig_activity, ax_activity = plt.subplots(figsize=(8, 6))
biomarker_cols_activity = [
    'Step_count',
    'Calories_burned',
    'Heart_Rate_bpm',
    'Average Sleep Duration (hrs)',
    'Sleep Quality (1-10)',
    '% with Sleep Disturbances'
]
correlation_matrix_activity = combined_df[biomarker_cols_activity].corr()
sns.heatmap(
    correlation_matrix_activity,
    annot=True,
    fmt='.2f',
    cmap='RdYlBu_r',
    center=0,
    ax=ax_activity,
    cbar_kws={'label': 'Correlation Coefficient'}
)
ax_activity.set_title('Physical Activity & Sleep Quality Correlation')
st.pyplot(fig_activity)
st.caption("Figure 4: Correlation matrix showing the associations between physical activity metrics (steps, calories burned, heart rate) and sleep quality measures - revealing potential lifestyle interactions affecting sleep outcomes.")

DeltaGenerator()

<!-- Old Maximum Glucose KPI cell removed because dashboard layout is now handled in a single Streamlit dashboard cell. -->

In [22]:
# Old cell replaced by the new Streamlit dashboard cell above.
st.markdown("---")
st.markdown("## Conclusion")
st.markdown(
    "This comprehensive dashboard provides an in-depth analysis of the complex relationships between sleep quality, blood glucose control, insulin management, physical activity, and demographic factors in individuals with diabetes. The visualizations and correlation analyses offer valuable insights for healthcare providers to optimize treatment strategies and improve patient outcomes."
)

DeltaGenerator()

<!-- Old Hyperglycemia KPI notes removed because KPI display is handled in the new dashboard cell. -->

In [7]:
# Old cell replaced by the new Streamlit dashboard cell above.
pass

<!-- Old scatter plot section removed: chart is now shown inside the Streamlit dashboard layout above. -->

In [8]:
# Old chart cell replaced by the new Streamlit dashboard cell above.
pass

<!-- Old heatmap section removed: correlation heatmap is now displayed in the Streamlit dashboard layout above. -->

In [9]:
# Old heatmap cell replaced by the new Streamlit dashboard cell above.
pass

<!-- Old histogram section removed: sleep quality distribution is now shown inside the Streamlit dashboard above. -->

In [10]:
# Old histogram cell replaced by the new Streamlit dashboard cell above.
pass

<!-- Old bar chart section removed: sleep disturbance comparison is now shown inside the Streamlit dashboard above. -->

In [11]:
# Old bar chart cell replaced by the new Streamlit dashboard cell above.
pass